<a href="https://colab.research.google.com/github/jtista/Online-Retail-II-Dissertation/blob/main/Online_Retail_II_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### SQL Database

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

In [3]:
FOLDER = '/content/drive/MyDrive/Online Retail II Dissertation/'

rfm = pd.read_csv(FOLDER + 'rfm_clusters.csv')
df  = pd.read_csv(FOLDER + 'transactions_clean.csv',
                  parse_dates=['InvoiceDate'])

print(f'rfm loaded: {len(rfm):,} rows')
print(f'df loaded:  {len(df):,} rows')
print(rfm['Segment'].value_counts())

rfm loaded: 3,969 rows
df loaded:  370,929 rows
Segment
At-Risk            2377
Lost Customers      918
Loyal Customers     576
Champions            98
Name: count, dtype: int64


In [4]:
print(rfm.shape)
print(rfm.columns.tolist())
print(rfm['Segment'].value_counts())

(3969, 9)
['Customer ID', 'Recency', 'Frequency', 'Monetary', 'CLV', 'Avg_Order_Value', 'Lifespan_Months', 'Cluster', 'Segment']
Segment
At-Risk            2377
Lost Customers      918
Loyal Customers     576
Champions            98
Name: count, dtype: int64


In [5]:
import sqlite3

conn = sqlite3.connect('segmentation.db')

# Save the rfm DataFrame to a SQL table named 'customers'
rfm.to_sql('customers', conn, if_exists='replace', index=False)
print(f"   Table 'customers' loaded: {len(rfm):,} rows\n")

   Table 'customers' loaded: 3,969 rows



CLV Ranking by Segment

In [6]:
query1 = """
    SELECT
        Segment,
        COUNT(*)                        AS Total_Customers,
        ROUND(AVG(CLV), 2)              AS Avg_CLV,
        ROUND(MIN(CLV), 2)              AS Min_CLV,
        ROUND(MAX(CLV), 2)              AS Max_CLV,
        ROUND(AVG(Monetary), 2)         AS Avg_Total_Spend,
        ROUND(AVG(Frequency), 1)        AS Avg_Invoices,
        ROUND(AVG(Recency), 1)          AS Avg_Recency_Days,
        ROUND(AVG(Avg_Order_Value), 2)  AS Avg_Order_Value,
        ROUND(AVG(Lifespan_Months), 1)  AS Avg_Lifespan_Months
    FROM customers
    GROUP BY Segment
    ORDER BY Avg_CLV DESC
"""
result1 = pd.read_sql(query1, conn)
print(result1.to_string(index=False))

        Segment  Total_Customers  Avg_CLV  Min_CLV   Max_CLV  Avg_Total_Spend  Avg_Invoices  Avg_Recency_Days  Avg_Order_Value  Avg_Lifespan_Months
      Champions               98 26577.02  4782.88 349164.35         15092.75          24.4              14.9           911.00                 11.3
Loyal Customers              576  4021.63   372.86  13544.99          4021.63          10.4              26.6           469.57                  9.7
        At-Risk             2377   864.11     3.75   4511.76           864.11           2.8              49.3           321.59                  4.7
 Lost Customers              918   479.04     2.95  10953.50           479.04           1.6             247.7           302.12                  1.7


Revenue Contribution by Segment

In [8]:
query2 = """
    SELECT
        Segment,
        COUNT(*)                            AS Total_Transactions,
        COUNT(DISTINCT "Customer ID")       AS Unique_Customers,
        ROUND(SUM(TotalSpend), 2)           AS Total_Revenue,
        ROUND(AVG(TotalSpend), 2)           AS Avg_Transaction_Value,
        ROUND(AVG(Price), 2)                AS Avg_Unit_Price,
        ROUND(AVG(Quantity), 1)             AS Avg_Quantity,
        ROUND(
            SUM(TotalSpend) * 100.0 /
            (SELECT SUM(TotalSpend) FROM transactions),
        2)                                  AS Revenue_Pct
    FROM transactions
    WHERE Segment IS NOT NULL
    GROUP BY Segment
    ORDER BY Total_Revenue DESC
"""

# Merge df with rfm to get the 'Segment' column for transaction data
transactions_with_segments = df.merge(rfm[['Customer ID', 'Segment']], on='Customer ID', how='left')

# Save the merged DataFrame to a SQL table named 'transactions'
transactions_with_segments.to_sql('transactions', conn, if_exists='replace', index=False)
print(f"   Table 'transactions' loaded: {len(transactions_with_segments):,} rows\n")

result2 = pd.read_sql(query2, conn)
print(result2.to_string(index=False))

   Table 'transactions' loaded: 370,929 rows

        Segment  Total_Transactions  Unique_Customers  Total_Revenue  Avg_Transaction_Value  Avg_Unit_Price  Avg_Quantity  Revenue_Pct
      Champions               63981                98     2604548.34                  40.71            3.36          24.4        35.13
Loyal Customers              131425               576     2316458.39                  17.63            3.08          10.8        31.24
        At-Risk              148859              2377     2053988.00                  13.80            2.99           8.3        27.70
 Lost Customers               26664               918      439761.22                  16.49            3.83           9.0         5.93


Top 3 products per segment

In [9]:
query3 = """
    SELECT
        Segment,
        Description,
        COUNT(*)                    AS Times_Purchased,
        ROUND(SUM(TotalSpend), 2)   AS Total_Revenue,
        ROUND(AVG(Price), 2)        AS Avg_Price,
        ROUND(AVG(Quantity), 1)     AS Avg_Quantity
    FROM transactions
    WHERE Segment IS NOT NULL
      AND Description IS NOT NULL
    GROUP BY Segment, Description
    ORDER BY Segment, Total_Revenue DESC
"""
result3     = pd.read_sql(query3, conn)
top_products = result3.groupby('Segment').head(3).reset_index(drop=True)
print(top_products.to_string(index=False))

        Segment                        Description  Times_Purchased  Total_Revenue  Avg_Price  Avg_Quantity
        At-Risk WHITE HANGING HEART T-LIGHT HOLDER             1071       32731.15       2.90          11.1
        At-Risk           REGENCY CAKESTAND 3 TIER              572       26957.70      12.63           3.9
        At-Risk      ASSORTED COLOUR BIRD ORNAMENT              513       13770.83       1.69          16.2
      Champions           REGENCY CAKESTAND 3 TIER              317       62997.75      12.02          17.8
      Champions WHITE HANGING HEART T-LIGHT HOLDER              583       49971.81       2.72          33.3
      Champions      ASSORTED COLOUR BIRD ORNAMENT              206       32375.53       1.61         102.9
 Lost Customers                             Manual               53       15778.75     294.55           5.9
 Lost Customers WHITE HANGING HEART T-LIGHT HOLDER              249        7026.80       2.90          10.2
 Lost Customers         VINT

Monthly Revenue by Segment

In [10]:
query4 = """
    SELECT
        Segment,
        SUBSTR(InvoiceDate, 1, 7)           AS Month,
        COUNT(*)                            AS Transactions,
        COUNT(DISTINCT "Customer ID")       AS Active_Customers,
        ROUND(SUM(TotalSpend), 2)           AS Monthly_Revenue,
        ROUND(AVG(TotalSpend), 2)           AS Avg_Transaction_Value
    FROM transactions
    WHERE Segment IS NOT NULL
    GROUP BY Segment, Month
    ORDER BY Segment, Month
"""

result4 = pd.read_sql(query4, conn)
print(result4.head(20).to_string(index=False))

  Segment   Month  Transactions  Active_Customers  Monthly_Revenue  Avg_Transaction_Value
  At-Risk 2009-12          7539               307        102928.05                  13.65
  At-Risk 2010-01          5400               216         70965.05                  13.14
  At-Risk 2010-02          6111               219         75619.63                  12.37
  At-Risk 2010-03          8749               355        125978.48                  14.40
  At-Risk 2010-04          7069               312        106035.08                  15.00
  At-Risk 2010-05          7407               340        104004.92                  14.04
  At-Risk 2010-06          9714               363        128777.53                  13.26
  At-Risk 2010-07          8823               382        130578.06                  14.80
  At-Risk 2010-08          9954               455        153304.36                  15.40
  At-Risk 2010-09         14769               622        237244.91                  16.06
  At-Risk 

Price sensitivity Preview per Segment

In [11]:
query5 = """
    SELECT
        Segment,
        ROUND(AVG(Price), 2)            AS Avg_Unit_Price,
        ROUND(MIN(Price), 2)            AS Min_Price,
        ROUND(MAX(Price), 2)            AS Max_Price,
        ROUND(AVG(Quantity), 2)         AS Avg_Quantity,
        ROUND(SUM(TotalSpend), 2)       AS Total_Revenue,
        COUNT(*)                        AS Total_Transactions
    FROM transactions
    WHERE Segment IS NOT NULL
    GROUP BY Segment
    ORDER BY Avg_Unit_Price DESC
"""

result5 = pd.read_sql(query5, conn)
print(result5.to_string(index=False))

        Segment  Avg_Unit_Price  Min_Price  Max_Price  Avg_Quantity  Total_Revenue  Total_Transactions
 Lost Customers            3.83        0.0   10953.50          8.98      439761.22               26664
      Champions            3.36        0.0   10468.80         24.38     2604548.34               63981
Loyal Customers            3.08        0.0    2667.88         10.76     2316458.39              131425
        At-Risk            2.99        0.0    1343.44          8.30     2053988.00              148859


Higher avg price segments are expected to be less price sensitive

In [15]:
import shutil, os

result1.to_csv(FOLDER + 'sql_clv_ranking.csv',     index=False)
result2.to_csv(FOLDER + 'sql_revenue.csv',         index=False)
result3.to_csv(FOLDER + 'sql_top_products.csv',    index=False)
result4.to_csv(FOLDER + 'sql_monthly_revenue.csv', index=False)
result5.to_csv(FOLDER + 'sql_price_analysis.csv',  index=False)

if os.path.exists('sql_analysis_charts.png'):
    shutil.copy('sql_analysis_charts.png', FOLDER)

shutil.copy('segmentation.db', FOLDER)


'/content/drive/MyDrive/Online Retail II Dissertation/segmentation.db'